### Agenda
- Learn to read csv files with headers and schema
- extract columns, and display
- filter datasets
- rename columns 
- select specific columns
- order by column values
- agg function count, min, max
- distinct

### Dataset
- San Francisco Fire Department Public Dataset
- https://www.kaggle.com/datasets/imankity/san-francisco-fire-department-public-dataset

### Analysis
- What were all the different types of fire calls in 2018?
- What months within the year 2018 saw the highest number of fire calls?
- Which neighborhood in San Francisco generated the most fire calls in 2018
- Which neighborhoods had the worst response times to fire calls in 2018
- Which week in the year in 2018 had the most fire calls?
- Is there a correlation between neighborhood, zip code, and number of fire calls

In [0]:
from pyspark.sql import functions as F

In [0]:
df = (spark
      .read
      .format("csv")
      .option("header", True)
      .option("inferSchama", True)
      .load("/Volumes/workspace/default/external_datasets/sf-fire-calls.csv")
)
df.printSchema()

In [0]:
df.columns

In [0]:
len(df.columns)

In [0]:
type(df.columns)

In [0]:
df.show(10, False)

In [0]:
df.display()

Databricks data profile. Run in Databricks to view.

In [0]:
few_df = (
    df.select("IncidentNumber", "AvailableDtTm", "CallType")
    .where(F.col("CallType")!= "Medical Incident")
)
few_df.display()

Projections and filters

In [0]:
few_fire_df = (df
.select("IncidentNumber", "AvailableDtTm", "CallType") 
.where(F.col("CallType") != "Medical Incident"))


In [0]:
few_fire_df.display()

In [0]:
few_fire_df = (df
.select(["IncidentNumber", "AvailableDtTm", "CallType"]) 
.where(F.col("CallType") != "Medical Incident"))

few_fire_df.display()

Renaming, adding, and dropping columns. 

In [0]:
df.select('Delay').display()

In [0]:
renamed_df = (
    df.withColumn("ResponseDelayedinMins", F.col("Delay").cast("decimal"))
)

renamed_df.where(F.col("ResponseDelayedinMins") > 5).display()

Casting columns type

In [0]:
fire_ts_df = (
    renamed_df.withColumn("IncidentDate", F.to_timestamp(F.col("CallDate"), "MM/dd/yyyy"))
    .withColumn("OnWatchDate", F.to_timestamp(F.col("WatchDate"), "MM/dd/yyyy"))
    .withColumn("AvailableDtTS", F.to_timestamp(F.col("AvailableDtTm"), "MM/dd/yyyy hh:mm:ss a"))
    .drop(*["CallDate","WatchDate","AvailableDtTm"])
)

fire_ts_df.select("IncidentDate", "OnWatchDate", "AvailableDtTS").display()

In [0]:
(fire_ts_df
 .select(F.year('IncidentDate'))
 .distinct()
 .orderBy(F.year('IncidentDate'))
 .display())

Aggregations

In [0]:
(fire_ts_df
 .select("CallType")
 .where(F.col("CallType").isNotNull())
 .groupBy("CallType")
 .count()
 .orderBy("count", ascending=False)
 .display())

min(), max(), sum(), and avg()

In [0]:
(fire_ts_df.select(F.sum("NumAlarms"), F.avg("ResponseDelayedinMins"),
F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins"))
.display())

In [0]:
(fire_ts_df.groupBy("CallType").agg(F.avg("ResponseDelayedinMins"),
F.min("ResponseDelayedinMins"), F.max("ResponseDelayedinMins"))
.display())

What were all the different types of fire calls in 2018?

In [0]:
fire_ts_df_2018 = fire_ts_df.where("year(IncidentDate) >= 2018")
fire_ts_df_2018.display()

In [0]:
fire_ts_df_2018.select("callType").distinct().display()

 What months within the year 2018 saw the highest number of fire calls?

In [0]:
(fire_ts_df_2018
 .select(F.month("IncidentDate").alias("IncidentMonth"),"CallType")
 .groupBy(F.col("IncidentMonth"))
 .agg(F.count("CallType").alias("count_of_incident"))
 .orderBy(F.col("count_of_incident").desc(), "IncidentMonth",)).display()

Which neighborhood in San Francisco generated the most fire calls in 2018

In [0]:
(fire_ts_df_2018
 .select(F.month("IncidentDate").alias("IncidentMonth"),"CallType", "Neighborhood")
 .groupBy(F.col("Neighborhood"))
 .agg(F.count("CallType").alias("count_of_incident"))
 .orderBy(F.col("count_of_incident").desc())).display()

Which neighborhoods had the worst response times to fire calls in 2018

In [0]:
(fire_ts_df_2018
 .select(F.month("IncidentDate").alias("IncidentMonth"),"CallType", "Neighborhood","ResponseDelayedinMins")
 .groupBy(F.col("Neighborhood"),F.col("CallType"))
 .agg(F.avg("ResponseDelayedinMins").alias("avg_ResponseDelayedinMins"), F.count("CallType").alias("Total_incident"))
 .orderBy(F.col("avg_ResponseDelayedinMins").desc())).display()

 Which week in the year in 2018 had the most fire calls?

In [0]:
(fire_ts_df_2018
 .select(F.month("IncidentDate").alias("IncidentMonth"), F.weekofyear("IncidentDate").alias("IncidentWeek"),"CallType")
 .groupBy(F.col("IncidentWeek"))
 .agg(F.count("CallType").alias("count_of_incident"))
 .orderBy(F.col("count_of_incident").desc(), "IncidentWeek",)).display()

Is there a correlation between neighborhood, zip code, and number of fire calls

In [0]:
output_df = (fire_ts_df_2018
 .select("IncidentDate","CallType", "Neighborhood","ResponseDelayedinMins", "zipcode")
 .groupBy(*["zipcode", "Neighborhood"])
 .agg(F.avg("ResponseDelayedinMins").alias("avg_ResponseDelayedinMins"), F.count("CallType").alias("Total_incident"))
 .orderBy(F.col("avg_ResponseDelayedinMins").desc()))

In [0]:
output_df.display()

• How can we use Parquet files or SQL tables to store this data and read it back?